In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"
# If charts do not display in Colab, uncomment:
# pio.renderers.default = "colab"

print("Plotly version:", __import__("plotly").__version__)

Plotly version: 5.24.1


In [3]:
# Load the dataset first
df = pd.read_excel('/content/bank_marketing_raw.xlsx')

# Data types and missing values per column
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(1)
})
summary

,dtype,missing,missing_%
age,int64,0,0.0
job,object,0,0.0
marital,object,0,0.0
education,object,0,0.0
default,object,0,0.0
balance,int64,0,0.0
housing,object,0,0.0
loan,object,0,0.0
contact,object,0,0.0
day,int64,0,0.0


In [4]:
# Summary statistics for important numerical columns
df[["age", "balance", "duration", "campaign", "pdays", "previous"]].describe().round(2)

,age,balance,duration,campaign,pdays,previous
count,999.00,999.00,999.00,999.00,999.00,999.00
mean,40.38,1293.18,846.07,2.96,1.21,0.03
std,9.74,2318.18,422.29,2.70,18.70,0.27
min,23.00,-1129.00,11.00,1.00,-1.00,0.00
25%,32.00,94.00,569.00,1.00,-1.00,0.00
50%,39.00,517.00,768.00,2.00,-1.00,0.00
75%,48.00,1571.00,1044.00,4.00,-1.00,0.00
max,61.00,34646.00,3881.00,29.00,198.00,4.00


In [5]:
# Fill missing values without changing the original categories
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].fillna("Unknown")

numeric_cols = ["age", "balance", "duration", "campaign", "pdays", "previous"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

MONTH_ORDER = ["jan", "feb", "mar", "apr", "may", "jun",
               "jul", "aug", "sep", "oct", "nov", "dec"]
months_present = [m for m in MONTH_ORDER if m in df["month"].astype(str).str.lower().unique()]
df["month"] = df["month"].astype(str).str.lower()

HOUSING_ORDER = ["no", "yes", "Unknown"]
HOUSING_COLORS = {"no": "#2A9D8F", "yes": "#E76F51", "Unknown": "#9CA3AF"}

df = df.dropna(subset=numeric_cols).reset_index(drop=True)

print("Housing loan counts:")
print(df["housing"].value_counts())
print("\nMonth counts:")
print(df["month"].value_counts().reindex(months_present))

Housing loan counts:
housing
yes    510
no     489
Name: count, dtype: int64

Month counts:
month
may    200
jun    161
jul    295
aug    225
oct     39
nov     79
Name: count, dtype: int64


In [6]:
fig1 = px.histogram(
    df, x="age",
    title="Distribution of Customer Age",
    labels={"age": "Customer age"},
    color_discrete_sequence=["#457B9D"]
)
fig1.update_layout(
    yaxis_title="Number of customers",
    bargap=0.08,
    xaxis=dict(rangeslider=dict(visible=True, thickness=0.12), dtick=5)
)
fig1.show()

In [7]:
fig2 = px.scatter(
    df, x="balance", y="duration", color="housing",
    category_orders={"housing": HOUSING_ORDER},
    color_discrete_map=HOUSING_COLORS,
    hover_data=["age", "job", "marital", "loan", "campaign", "month"],
    opacity=0.75,
    labels={"balance": "Account balance", "duration": "Call duration (seconds)",
            "housing": "Housing loan"},
    title="Account Balance vs. Call Duration by Housing Loan Status"
)
fig2.show()

In [8]:
fig3 = px.box(
    df, x="housing", y="balance", color="housing",
    category_orders={"housing": HOUSING_ORDER},
    color_discrete_map=HOUSING_COLORS,
    points="all",
    hover_data=["age", "job", "marital", "duration", "campaign"],
    labels={"housing": "Housing loan", "balance": "Account balance"},
    title="Account Balance by Housing Loan Status (All Customers)"
)
fig3.update_traces(marker=dict(size=4, opacity=0.55), jitter=0.35)
fig3.update_layout(showlegend=False)
fig3.show()

In [9]:
# Numbers behind the box plot
df.groupby("housing")[["age", "balance", "duration", "campaign"]].median().round(2)

,age,balance,duration,campaign
housing,,,,
no,42.0,550.0,727.0,2.0
yes,36.0,481.0,797.0,2.0


In [10]:
fig4 = px.scatter_3d(
    df, x="age", y="balance", z="duration", color="housing",
    category_orders={"housing": HOUSING_ORDER},
    color_discrete_map=HOUSING_COLORS,
    hover_data=["job", "marital", "loan", "campaign", "month"],
    opacity=0.75,
    labels={"age": "Customer age", "balance": "Account balance",
            "duration": "Call duration (seconds)", "housing": "Housing loan"},
    title="Customer Age, Account Balance and Call Duration in 3D"
)
fig4.update_traces(marker=dict(size=3.5))
fig4.update_layout(height=650)
fig4.show()

In [11]:
# Keep month order chronological
animation_df = df.copy()
animation_df["month"] = pd.Categorical(
    animation_df["month"], categories=months_present, ordered=True
)
animation_df = animation_df.sort_values("month")

fig5 = px.scatter(
    animation_df, x="age", y="balance", color="housing",
    animation_frame="month",
    category_orders={"housing": HOUSING_ORDER, "month": months_present},
    color_discrete_map=HOUSING_COLORS,
    hover_data=["job", "marital", "loan", "duration", "campaign"],
    opacity=0.75,
    labels={"age": "Customer age", "balance": "Account balance",
            "housing": "Housing loan", "month": "Contact month"},
    title="Customer Age vs. Account Balance by Contact Month"
)
fig5.update_layout(legend_itemsizing="constant")
if fig5.layout.updatemenus and fig5.layout.updatemenus[0].buttons:
    fig5.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 800
fig5.show()

In [12]:
# Number of customers in each contact month
df.groupby("month", observed=False).size().reindex(months_present).rename("customers").to_frame().T

month,may,jun,jul,aug,oct,nov
customers,200,161,295,225,39,79


In [13]:
metrics = {
    "balance": ("Account balance", "linear"),
    "duration": ("Call duration (seconds)", "linear"),
    "campaign": ("Number of contacts in this campaign", "linear"),
    "previous": ("Previous contacts", "linear"),
}

fig6 = go.Figure()
fig6.add_trace(go.Scatter(
    x=df["age"], y=df["balance"], mode="markers",
    marker=dict(
        color=df["housing"].map(HOUSING_COLORS).fillna("#9CA3AF"),
        size=8, opacity=0.7, line=dict(width=0.5, color="white")
    ),
    text=df["job"],
    customdata=np.stack([df["housing"], df["marital"]], axis=-1),
    hovertemplate=(
        "Job: %{text}<br>Age: %{x}<br>Value: %{y}"
        "<br>Housing loan: %{customdata[0]}"
        "<br>Marital status: %{customdata[1]}<extra></extra>"
    ),
    name="Customers"
))

buttons = []
for col, (label, axis_type) in metrics.items():
    buttons.append(dict(
        label=col,
        method="update",
        args=[
            {"y": [df[col]]},
            {"yaxis.title.text": label,
             "yaxis.type": axis_type,
             "title.text": f"Customer Age vs. {col.title()}"}
        ]
    ))

fig6.update_layout(
    title="Customer Age vs. Account Balance",
    xaxis_title="Customer age",
    yaxis_title=metrics["balance"][0],
    updatemenus=[dict(
        buttons=buttons, direction="down",
        x=1.0, xanchor="right", y=1.15, yanchor="top", showactive=True
    )],
    annotations=[dict(
        text="Y-axis metric:", x=0.89, xref="paper", y=1.12, yref="paper",
        showarrow=False, xanchor="right"
    )]
)
fig6.show()

In [14]:
# Spearman correlations: compare relationships between numerical variables
df[["age", "balance", "duration", "campaign", "pdays", "previous"]].corr(
    method="spearman"
).round(2)

,age,balance,duration,campaign,pdays,previous
age,1.00,0.15,0.04,-0.00,0.01,0.01
balance,0.15,1.00,0.01,-0.03,0.05,0.05
duration,0.04,0.01,1.00,0.10,-0.08,-0.08
campaign,-0.00,-0.03,0.10,1.00,-0.10,-0.10
pdays,0.01,0.05,-0.08,-0.10,1.00,1.00
previous,0.01,0.05,-0.08,-0.10,1.00,1.00


In [15]:
figs = [fig1, fig2, fig3, fig4, fig5, fig6]
with open("Bank_Activity_interactive_charts.html", "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'><title>Bank Activity Interactive Charts</title></head><body>")
    for i, fig in enumerate(figs):
        f.write(fig.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False,
                            auto_play=False))
    f.write("</body></html>")
print("Saved Bank_Activity_interactive_charts.html")

Saved Bank_Activity_interactive_charts.html
